# BLIP Caption Fine-Tuning (local)

Fine-tunes `Salesforce/blip-image-captioning-base` on a small custom
"daily-life" caption dataset so the generated captions reflect **actions,
emotions and atmosphere** (the signal we use for music recommendation).

**Pipeline step:** this notebook is *BLIP Fine-tuning* in the system pipeline.
It produces `./blip_best/`, which `video_to_music_recommendation.py` then loads.

### What you need before running
```
video_to_music_recommender/
  data/
    train.json          # [{"image": "images/001.jpg", "caption": "..."}, ...]
    val.json
    images/             # the jpg/png files referenced above
```
Image paths inside the json may be absolute, or relative to the `data/` folder.

Run the cells top to bottom. On a CPU-only machine keep the dataset small
(a few hundred images) or training will be slow.

In [ ]:
# Install dependencies (run once). Uncomment if needed.
# %pip install -r requirements.txt

In [ ]:
import os

# ----- paths -----
BASE_DIR   = os.getcwd()                              # run the notebook from the project folder
DATA_DIR   = os.path.join(BASE_DIR, "data")
TRAIN_JSON = os.path.join(DATA_DIR, "train.json")
VAL_JSON   = os.path.join(DATA_DIR, "val.json")
IMAGE_ROOT = DATA_DIR                                 # relative image paths resolve against this
OUTPUT_DIR = os.path.join(BASE_DIR, "blip_best")      # <- consumed by the app

# ----- model / training config -----
BASE_MODEL            = "Salesforce/blip-image-captioning-base"
NUM_EPOCHS            = 5
BATCH_SIZE            = 4
LR                    = 5e-5
MAX_TEXT_LEN          = 64
FREEZE_VISION_ENCODER = True    # per the report: freeze vision encoder, tune the text decoder
NUM_WORKERS           = 0       # keep 0 on Windows
SEED                  = 42

In [ ]:
import json, random
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

random.seed(SEED)
torch.manual_seed(SEED)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else "cpu"
)
print("device:", device)

In [ ]:
class JsonCaptionDataset(Dataset):
    """Reads a list of {"image": ..., "caption": ...} records."""

    def __init__(self, json_path: str, image_root: str | None = None):
        with open(json_path, "r", encoding="utf-8") as f:
            self.items = json.load(f)
        self.image_root = image_root

    def _resolve(self, p: str) -> str:
        if os.path.isabs(p) or self.image_root is None:
            return p
        return os.path.join(self.image_root, p)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        it = self.items[idx]
        image_path = self._resolve(it["image"])
        caption = it["caption"]
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Missing image: {image_path}")
        image = Image.open(image_path).convert("RGB")
        return {"image": image, "caption": caption, "image_path": image_path}


@dataclass
class BlipCollator:
    processor: BlipProcessor
    max_text_len: int = 64

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        images   = [b["image"] for b in batch]
        captions = [b["caption"] for b in batch]

        enc_img = self.processor(images=images, return_tensors="pt")
        enc_txt = self.processor.tokenizer(
            captions,
            padding=True,
            truncation=True,
            max_length=self.max_text_len,
            return_tensors="pt",
        )
        labels = enc_txt["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100  # ignore padding in the loss

        return {
            "pixel_values": enc_img["pixel_values"],
            "input_ids": enc_txt["input_ids"],
            "attention_mask": enc_txt["attention_mask"],
            "labels": labels,
            "captions": captions,
        }

In [ ]:
processor = BlipProcessor.from_pretrained(BASE_MODEL)

train_ds = JsonCaptionDataset(TRAIN_JSON, image_root=IMAGE_ROOT)
val_ds   = JsonCaptionDataset(VAL_JSON,   image_root=IMAGE_ROOT)
collator = BlipCollator(processor=processor, max_text_len=MAX_TEXT_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collator, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collator, num_workers=NUM_WORKERS)

print("train size:", len(train_ds), "| val size:", len(val_ds))

In [ ]:
model = BlipForConditionalGeneration.from_pretrained(BASE_MODEL).to(device)

if FREEZE_VISION_ENCODER:
    for p in model.vision_model.parameters():
        p.requires_grad = False

trainable = [p for p in model.parameters() if p.requires_grad]
print("trainable params:", f"{sum(p.numel() for p in trainable):,}")
optimizer = torch.optim.AdamW(trainable, lr=LR)

In [ ]:
@torch.no_grad()
def evaluate(loader):
    """Returns (mean cross-entropy loss, token-level accuracy) on a loader."""
    model.eval()
    total_loss, total_correct, total_tokens, n = 0.0, 0, 0, 0
    for batch in loader:
        pixel_values = batch["pixel_values"].to(device)
        input_ids    = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels       = batch["labels"].to(device)

        out = model(pixel_values=pixel_values, input_ids=input_ids,
                    attention_mask=attention_mask, labels=labels)
        bs = labels.size(0)
        total_loss += out.loss.item() * bs
        n += bs

        # next-token prediction: align logits[t] with labels[t+1]
        pred = out.logits[:, :-1, :].argmax(-1)
        gold = labels[:, 1:]
        mask = gold != -100
        total_correct += (pred[mask] == gold[mask]).sum().item()
        total_tokens  += mask.sum().item()

    return total_loss / max(n, 1), total_correct / max(total_tokens, 1)

In [ ]:
best_val_loss = float("inf")
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running, steps = 0.0, 0
    for step, batch in enumerate(train_loader, 1):
        pixel_values   = batch["pixel_values"].to(device)
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        out = model(pixel_values=pixel_values, input_ids=input_ids,
                    attention_mask=attention_mask, labels=labels)
        loss = out.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()

        running += loss.item(); steps += 1
        if step % 10 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)}  loss {running/steps:.4f}")

    train_loss = running / max(steps, 1)
    val_loss, val_acc = evaluate(val_loader)
    history.append({"epoch": epoch, "train_loss": train_loss,
                    "val_loss": val_loss, "val_token_acc": val_acc})
    print(f"[epoch {epoch}] train_loss={train_loss:.4f}  "
          f"val_loss={val_loss:.4f}  val_token_acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        model.save_pretrained(OUTPUT_DIR)
        processor.save_pretrained(OUTPUT_DIR)
        print(f"    -> saved best model to {OUTPUT_DIR}")

with open(os.path.join(BASE_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
# Training curves (loss + token-level accuracy)
try:
    import matplotlib.pyplot as plt
    ep = [h["epoch"] for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(ep, [h["train_loss"] for h in history], marker="o", label="train")
    ax[0].plot(ep, [h["val_loss"] for h in history], marker="o", label="val")
    ax[0].set_xlabel("epoch"); ax[0].set_title("Cross-entropy loss"); ax[0].legend()
    ax[1].plot(ep, [h["val_token_acc"] for h in history], marker="o", color="green")
    ax[1].set_xlabel("epoch"); ax[1].set_title("Val token-level accuracy")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("skip plot:", e)

In [ ]:
# Sanity check: generate a caption with the fine-tuned model
best_processor = BlipProcessor.from_pretrained(OUTPUT_DIR)
best_model = BlipForConditionalGeneration.from_pretrained(OUTPUT_DIR).to(device).eval()

sample = val_ds[0]
inputs = best_processor(images=sample["image"], return_tensors="pt").to(device)
with torch.no_grad():
    gen = best_model.generate(**inputs, max_new_tokens=30, num_beams=3)

print("GOLD:", sample["caption"])
print("PRED:", best_processor.decode(gen[0], skip_special_tokens=True))

Done. `./blip_best/` now holds the fine-tuned model.

Next: from the project folder run
```
streamlit run app.py
```